In [15]:
import pandas as pd
import numpy as np
from numpy.random import Generator, PCG64
from joblib import Parallel, delayed
import altair as alt

rng = Generator(PCG64())

In [6]:
all_df = pd.read_csv(
    "https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/final-data/all_valid_responses_df.csv",
    low_memory=False
)
all_df['item_id'] = all_df['question'] + all_df['image_file']

In [16]:
model_configs = [
    {'model': "Salesforce/blip2-flan-t5-xl", 'top_p': 'p06', 'temp': 't1', 'color': '#5ba3cf', 'accent_color': '#8cbedd'},
    {'model': "Salesforce/blip2-flan-t5-xxl", 'top_p': 'p1', 'temp': 't10', 'color': '#4c78a8', 'accent_color': '#81a0c2'},
    {'model': "llava-hf/llava-1.5-7b-hf", 'top_p': 'p04', 'temp': 't1', 'color': '#f9b574', 'accent_color': '#facb9d'},
    {'model': "llava-hf/llava-1.5-13b-hf", 'top_p': 'p1', 'temp': 't04', 'color': '#f58518', 'accent_color': '#f79d46'},
    {'model': "liuhaotian/llava-v1.6-34b", 'top_p': 'p1', 'temp': 't04', 'color': '#BF9000', 'accent_color': '#F1C232'},
    {'model': "google/pix2struct-chartqa-base", 'top_p': 'p08', 'temp': 't1', 'color': '#ba94c4', 'accent_color': '#c7b8d9'},
    {'model': "google/matcha-chartqa", 'top_p': 'p04', 'temp': 't1', 'color': '#8b6db2', 'accent_color': '#a28ac1'},
    {'model': "GPT-4V", 'top_p': 'p1', 'temp': 't02', 'color': '#c93739', 'accent_color': '#C6765E'},
    {'model': "Human", 'top_p': 'pna', 'temp': 'tna', 'color': '#2e693b', 'accent_color': '#73D287'},
]

### Find Human-Model Correlation

In [218]:
def bootstrap_correlation(
    model_responses, 
    human_responses,
    n_iterations=1000,
):
    """Added bootstrap coorelation """

    def bootstrap_iteration(
        model_responses,
        human_responses,
        chosen_items
    ):
        model_filter_df = pd.concat(
            [model_responses[model_responses['item_id'] == item] for item in chosen_items]
        )

        huamn_filter_df = pd.concat(
            [human_responses[human_responses['item_id'] == item] for item in chosen_items]
        )

        item_level_model_responses = model_filter_df.groupby(
            ['item_id', 'test_type']
        )['is_correct'].mean().reset_index().sort_values(by='item_id')

        item_level_human_responses = huamn_filter_df.groupby(
            ['item_id', 'test_type']
        )['is_correct'].mean().reset_index().sort_values(by='item_id')

        if item_level_model_responses['is_correct'].sum() == 0:
            human_model_correlation = np.nan
        else:
            human_model_correlation = np.corrcoef(
                item_level_model_responses['is_correct'],
                item_level_human_responses['is_correct']
            )[1][0]
            
        return human_model_correlation

    items = model_responses['item_id'].unique()
    bootstrapped_correlations = Parallel(n_jobs=-1)(
        delayed(bootstrap_iteration)(
            model_responses,
            human_responses,
            rng.choice(items, len(items), replace=True)
        ) for _ in range(n_iterations)
    )

    # get rid of all NA values
    bootstrapped_correlations = [bc for bc in bootstrapped_correlations if not pd.isna(bc)]

    if len(bootstrapped_correlations) == 0:
        return [np.nan, np.nan]
        
    lower = np.percentile(bootstrapped_correlations, 2.5)
    upper = np.percentile(bootstrapped_correlations, 97.5)

    return [lower, upper]

def human_model_correlation(all_df):
    """
    Find correlation on model and human responses on 

    """
    
    # grouped_df = all_df.groupby(['item_id', 'agent_type', 'test_type'])['is_correct'].mean().reset_index()
    human_model_corr_df = []

    for test in all_df['test_type'].unique():
        df = all_df[all_df['test_type'] == test].copy()
        
        human_responses = df[df['agent_type'] == 'Human']
        for agent in all_df['agent_type'].unique():
            if agent == 'Human':
                continue
            else:
                model_responses = df[df['agent_type'] == agent].copy()
                human_responses_valid = human_responses[
                    human_responses['item_id'].isin(model_responses['item_id'].unique())
                ]
            
            item_level_model_responses = model_responses.groupby(
                ['item_id', 'test_type']
            )['is_correct'].mean().reset_index().sort_values(by='item_id')

            item_level_human_responses = human_responses_valid.groupby(
                ['item_id', 'test_type']
            )['is_correct'].mean().reset_index().sort_values(by='item_id')

            # if the model only has incorrect responses for all questions
            if item_level_model_responses['is_correct'].sum() == 0:
                human_model_correlation = np.nan
            else:
                human_model_correlation = np.corrcoef(
                    item_level_model_responses['is_correct'],
                    item_level_human_responses['is_correct']
                )[1][0]

            lower, upper = bootstrap_correlation(
                model_responses,
                human_responses_valid
            )

            human_model_corr_df.append({
                'model': agent,
                'test': test,
                'lower_correlation': lower,
                'upper_correlation': upper,
                'correlation': human_model_correlation
            })

    return pd.DataFrame(human_model_corr_df)

try:
    print("reading human model correlation")
    human_model_correlation_df = pd.read_csv("./dataframes/human_model_correlation_df.csv")
except:
    print("general human model correlation")
    human_model_correlation_df = human_model_correlation(all_df)
    human_model_correlation_df.to_csv("./dataframes/human_model_correlation_df.csv")

reading human model correlation


In [219]:
# human_model_correlation_df = human_model_correlation(all_df)

In [221]:
# human_model_correlation_df

### Find split-half reliability

In [230]:
def bootstrap_pairwise_correlation(
    human_responses,
    n_iterations=1000,
):
    """Split-half pairwise correlation bootstrapped over human responses"""

    def bootstrap_iteration(
        human_responses,
        chosen_items
    ):
        huamn_filter_df = pd.concat(
            [human_responses[human_responses['item_id'] == item] for item in chosen_items]
        ).sample(frac=1)

        human_half1 = huamn_filter_df.iloc[0:len(huamn_filter_df)//2]
        human_half2 = huamn_filter_df.iloc[len(huamn_filter_df)//2:]

        human_half1 = human_half1[human_half1['item_id'].isin(human_half2['item_id'])]
        human_half2 = human_half2[human_half2['item_id'].isin(human_half1['item_id'])]

        item_level_human_half1_responses = human_half1.groupby(
            ['item_id', 'test_type']
        )['is_correct'].mean().reset_index().sort_values(by='item_id')

        item_level_human_half2_responses = human_half2.groupby(
            ['item_id', 'test_type']
        )['is_correct'].mean().reset_index().sort_values(by='item_id')

        human_model_correlation = np.corrcoef(
            item_level_human_half1_responses['is_correct'],
            item_level_human_half2_responses['is_correct']
        )[1][0]
            
        return human_model_correlation

    items = human_responses['item_id'].unique()
    bootstrapped_correlations = Parallel(n_jobs=-1)(
        delayed(bootstrap_iteration)(
            human_responses,
            rng.choice(items, len(items), replace=True)
        ) for _ in range(n_iterations)
    )

    bootstrapped_correlations = [bc for bc in bootstrapped_correlations if not pd.isna(bc)]

    if len(bootstrapped_correlations) == 0:
        return [np.nan, np.nan]
        
    lower = np.percentile(bootstrapped_correlations, 2.5)
    upper = np.percentile(bootstrapped_correlations, 97.5)
    mean = np.mean(bootstrapped_correlations)

    return [mean, lower, upper]

def human_human_correlation(all_df):
    """
    Find correlation on model and human responses on 

    """
    
    human_model_corr_df = []

    for test in all_df['test_type'].unique():
        df = all_df[all_df['test_type'] == test].copy()
        
        human_responses = df[df['agent_type'] == 'Human']
        
        mean, lower, upper = bootstrap_pairwise_correlation(
            human_responses
        )
        human_model_corr_df.append({
            'model': 'Human',
            'test': test,
            'lower_correlation': lower,
            'upper_correlation': upper,
            'correlation': mean
        })

    return pd.DataFrame(human_model_corr_df)

try:
    print("reading human model correlation")
    human_model_correlation_df = pd.read_csv("./dataframes/human_model_correlation_df.csv")
except:
    print("general human model correlation")
    human_human_correlation_df = human_model_correlation(all_df)
    human_human_correlation_df.to_csv("./dataframes/human_human_correlation_df.csv")

reading human model correlation


In [232]:
# human_human_correlation_df

### Correlation Error Bars

In [245]:
def create_graph_performance_chart(correlation_df, tests, domain_order, domain_color=[]):
    """ Generate chart for point estimate and confidence intervals overlayed with item level scatteplots
    """
    charts = []
    for test in tests:
        df = correlation_df[correlation_df['test'] == test].copy()
        
        mean_point_plot = alt.Chart(df).mark_point(filled=True, size=50, opacity=1).encode(
            x=alt.X('model:N', title='Category', scale=alt.Scale(domain=domain_order)),
            y=alt.Y('correlation:Q', title='', scale=alt.Scale(domain=[-1, 1])),
            color=alt.Color('model:N', scale=alt.Scale(domain=domain_order, range=domain_color), legend=None),
        )
        
        error_bars = alt.Chart(df).mark_rule(strokeWidth=2).encode(
            x=alt.X('model:N'),
            y=alt.Y('lower_correlation:Q'),
            y2='upper_correlation:Q',
            color=alt.Color('model:N', scale=alt.Scale(domain=domain_order, range=domain_color), legend=None)
        )

        split_half_band = alt.Chart(df[df['model'] == 'Human']).mark_rect(color='black', opacity=0.2).encode(
            y=alt.Y(f'lower_correlation:Q'),
            y2=(f'upper_correlation:Q'),
        )
    
        split_half_mean = alt.Chart(df[df['model'] == 'Human']).mark_rule(color='black', strokeDash=[5, 5]).encode(
            y=alt.Y(f'correlation:Q'),
            strokeWidth=alt.value(1)
        )
        
        chart = (mean_point_plot + error_bars + split_half_band + split_half_mean).properties(height=200, width=100, title=test)

        charts.append(chart)


    return alt.hconcat(*charts) #.resolve_scale(color='independent')

tests = ['ggr', 'vlat', 'calvi', 'holf', 'holf2', 'chartqa-test-continuous-human']
domain_order = [mc['model'] for mc in model_configs if mc['model'] != 'Human']
domain_color = [mc['color'] for mc in model_configs if mc['model'] != 'Human']


create_graph_performance_chart(
    pd.concat([
        human_model_correlation_df,
        human_human_correlation_df
    ]),
    tests=tests,
    domain_order=domain_order, 
    domain_color=domain_color,
) 

alt.HConcatChart(...)

In [252]:
# all_df[all_df['test_type'] == 'chartqa-test-continuous-human'].sort_values(by=['is_correct'])